# 02 - Synthetic data generation

## Objective

The objective of this notebook is to design and implement a synthetic data generator for industrial electrical coils.

The generator will simulate a simplified manufacturing process based on:

- product characteristics;
- process parameters;
- environmental conditions;
- physical and process assumptions defined in the domain analysis.

Each generated observation represents one manufactured coil.

The synthetic dataset will later be used for:

- exploratory data analysis;
- predictive modelling;
- anomaly detection;
- explainability experiments.

## Coil production representation

```text
Product characteristics
       │
       ▼
Process parameters ───────────┐
                              │
Environmental conditions ─────┤
                              ▼
                     Manufacturing model
                              │
                              ▼
                   Process quality indicators
                              │
              ┌───────────────┼────────────────┬───────────────┐
              ▼               ▼                ▼               ▼
         Resistance        Rigidity    Insulation resistance  Displacement
              │               │                │               │
              └───────────────┴────────────────┴───────────────┘
                              │
                              ▼
                        PASS / FAIL

In more detail:

Product characteristics
│
├── Product family
├── Coil geometry
├── Wire cross-section
└── Nominal number of turns

Process quality indicators
│
├── Winding quality
├── Winding regularity
├── Mechanical stress
└── Welding quality
```

## Electric displacement: definition and dimensional analysis

The electric displacement field is defined as:

$$
\mathbf{D} = \varepsilon_0\mathbf{E} + \mathbf{P}
$$

where:

- $\mathbf{D}$ = electric displacement field
- $\varepsilon_0$ = vacuum permittivity
- $\mathbf{E}$ = electric field
- $\mathbf{P}$ = electric polarization

For a linear dielectric material:

$$
\mathbf{D} = \varepsilon\mathbf{E}
$$

with:

$$
\varepsilon = \varepsilon_r\varepsilon_0
$$

In more detail:

$$
\mathbf{P} = \varepsilon_0\chi_e\mathbf{E}
$$

where:

$$
\chi_e = \text{electric susceptibility (dimensionless)}
$$

$$
\chi_e = \varepsilon_r - 1
$$

$$
\begin{aligned}
\mathbf{D} &= \varepsilon_0\mathbf{E} + \mathbf{P} \\
            &= \varepsilon_0\mathbf{E} + \varepsilon_0\chi_e\mathbf{E} \\
            &= \varepsilon_0(1+\chi_e)\mathbf{E} \\
            &= \varepsilon_0\varepsilon_r\mathbf{E} \\
            &= \varepsilon\mathbf{E}
\end{aligned}
$$

### Dimensional analysis: expression in fundamental SI units

The electric displacement field has units:
$$
[D] = \frac{C}{m^2}
$$

Since:

$$
C = A \cdot s
$$

we obtain:

$$
[D] = A\,s\,m^{-2}
$$

For the vacuum permittivity:

$$
[N] = kgms^{-2}
$$

Therefore:

$$
\begin{aligned}
\left[\varepsilon_0\right] &= \frac{C^2}{N\,m^2} \\
                           &= \frac{(A\,s)^2}{(kg\,m\,s^{-2})m^2} \\
                           &= kg^{-1}m^{-3}s^4A^2
\end{aligned}
$$

## Resistivity curves for different RRR values

<p align="center">
  <img src="../data/synthetic/resistivity_RRR.png" width="700">
</p>

## Resistivity: mathematical modelling

We can write the resistivity in the following way:

$$
\rho(T) = \rho_i(T) + \rho_0
$$

where:

- $\rho_i(T)$: intrinsic component (ideal), which depends on temperature;
- $\rho_0$: residual component, which is approximately constant and depends on the material quality;
- $RRR$: Residual Resistivity Ratio, which can be used as an indicator of material quality and to estimate the magnitude of $\rho_0$.

From the graph we have:

$$
RRR = \frac{\rho(273\,K)}{\rho(4\,K)}
$$

We can approximate:

$$
\rho_0 \approx \rho(4\,K)
$$

Therefore:

$$
\begin{aligned}
RRR
&= \frac{\rho(273\,K)}{\rho(4\,K)} \\
&\approx \frac{\rho(273\,K)}{\rho_0}
\end{aligned}
$$

and therefore:

$$
\boxed{
\rho_0 \approx \frac{\rho(273\,K)}{RRR}
}
$$

We can think of $RRR$ as a quality indicator:

$$
RRR \uparrow
\quad\Longrightarrow\quad
\rho_0 \downarrow
\quad\Longrightarrow\quad
\text{higher material purity}
$$

### First-order Taylor approximation of resistivity

Now we can write the first-order Taylor expansion for the resistivity, with the Peano remainder, centered at:

$$
T_0 = 273.15\,K
$$

We start from:

$$
\rho(T)=\rho_i(T)+\rho_0
$$

Therefore:

$$
\begin{aligned}
\rho(T)
&=
\rho_0+\rho_i(T_0)
+
\left.
\frac{d\rho_i}{dT}
\right|_{T_0}
(T-T_0)
+
o(T-T_0)
\end{aligned}
$$

From the graph we have:

$$
T_0=273.15\,K=0^\circ C
$$

and:

$$
\rho_i(0^\circ C)
=
1.545\times10^{-8}\,\Omega\cdot m
$$

as well as:

$$
\left.
\frac{d\rho_i}{dT}
\right|_{T_0}
=
6.7\times10^{-11}\,\Omega\cdot m/K
$$

Therefore, for the temperature $T_K$ in Kelvin:

$$
\rho(T_K)
=
\rho_0
+
1.545\times10^{-8}
+
6.7\times10^{-11}(T_K-273.15)
+
o(T_K-273.15)
$$

We know that:

$$
T_K=T_C+273.15
$$

therefore:

$$
T_C=T_K-273.15
$$

Finally, we obtain the first-order approximation in terms of Celsius temperature:

$$
\boxed{
\rho(T_C)
\approx
\rho_0
+
1.545\times10^{-8}
+
6.7\times10^{-11}T_C
}
$$

where $T_C$ is expressed in $^\circ C$.

### First-order approximation of $\Delta\rho$

We have that:

$$
\rho(T_0+dT_K)-\rho_i(T_0)
=
\rho_0
+
\rho_i'(T_0)dT_K
+
o(dT_K)
$$

as

$$
dT_K \to 0\,K.
$$

Synthetically:

$$
\Delta\rho(T_K)
=
\rho_0
+
\rho_i'(T_0)dT_K
+
o(dT_K),
\qquad
dT_K \to 0\,K.
$$

In conclusion:

$$
\Delta\rho(273.15\,K)
\approx
\rho_0
+
d\rho_i(273.15\,K)
$$

in a neighbourhood of $273.15\,K$.

Equivalently, in Celsius:

$$
\Delta\rho(0^\circ C)
\approx
\rho_0
+
d\rho_i(0^\circ C)
$$

in a neighbourhood of $0^\circ C$.

## Facts to be considered

1. The coils produced are either single or double winding, in small
   or medium series.

2. The copper wire has a diameter that ranges from $0.035$ mm
   to $2.00$ mm.

3. The windings can be produced with an air core or on round, square,
   or rectangular bobbins.

### Categorical variables

#### Winding configuration

The winding type is defined as:

$$
\text{Winding type}
\in
\{
\text{single winding},
\text{double winding}
\}.
$$

#### Winding support

The winding support is defined as:

$$
\text{Winding support}
\in
\{
\text{air core},
\text{round bobbin},
\text{square bobbin},
\text{rectangular bobbin}
\}.
$$

The core type is defined as:

$$
\text{Core type}
\in
\{\text{air core},\text{round bobbin},\text{square bobbin},
\text{rectangular bobbin}\}.
$$

### Wire diameter

The copper wire diameter $d$ is specified within the following range:

$$
\boxed{
0.035\ \mathrm{mm}
\leq
d
\leq
2.00\ \mathrm{mm}
}
$$

## Variables modelling

### Number of turns

We have the following schema:
```text
Coil geometry
      │
      ▼
Number of turns (N)
      │
      ▼
Wire length (L)
      │
      ├──────────────┐
      ▼              │
Resistivity (ρ)      │
      │              │
      └──────┬───────┘
             ▼
       Resistance (R)
```

#### Relationship between number of turns and wire length

The number of turns determines the total length of copper wire
through the coil winding.

In a first approximation:

$$
L \approx N\,l_{\text{turn}}
$$

where:

- $N$ is the number of turns;
- $L$ is the total length of copper wire;
- $l_{\text{turn}}$ is the average length of one turn.

Therefore:

$$
\boxed{N \longrightarrow L}
$$

#### From coil geometry to electrical resistance

The coil geometry determines the average length of one turn.
Therefore, together with the number of turns, it determines the
total length of copper wire:

$$
\text{Coil geometry}
\longrightarrow
l_{\text{turn}}
$$

$$
N \times l_{\text{turn}}
\longrightarrow
L
$$

The total wire length, together with the wire cross-section and
the temperature-dependent resistivity, determines the electrical
resistance:

$$
L,\ A,\ \rho(T)
\longrightarrow
R
$$

with:

$$
\boxed{
R=\rho(T)\frac{L}{A}
}
$$

#### Geometric calculation of the number of turns

The overall wire diameter includes both the copper conductor and
the enamel insulation.

Therefore:

$$
d_{\mathrm{outer}} > d_{\mathrm{Cu}}
$$

where:

- $d_{\mathrm{Cu}}$ is the bare copper conductor diameter;
- $d_{\mathrm{outer}}$ is the overall diameter of the enamelled wire.

The overall wire diameter is therefore used for the geometric
calculation of the winding.

For a cylindrical coil, we define:

- $h$ = axial length of the coil;
- $r$ = inner radius of the winding;
- $R$ = outer radius of the winding;
- $d_{\mathrm{outer}}$ = overall diameter of the enamelled wire.

The approximate number of turns per layer is:

$$
\boxed{
N_{\mathrm{per\ layer}}
\approx
\frac{h}{d_{\mathrm{outer}}}
}
$$

The approximate number of radial layers is:

$$
\boxed{
n_{\mathrm{layers}}
\approx
\frac{R-r}{d_{\mathrm{outer}}}
}
$$

Therefore, the approximate total number of turns is:

$$
N
\approx
N_{\mathrm{per\ layer}}
\,
n_{\mathrm{layers}}
$$

and hence:

$$
\boxed{
N
\approx
\frac{h(R-r)}
{d_{\mathrm{outer}}^2}
}
$$

This represents an initial geometric approximation. In a real
winding process, the actual number of turns also depends on factors
such as winding pitch, insulation thickness, packing factor,
manufacturing tolerances and winding configuration.

### From wire diameter to number of turns

The geometric relationship can be represented as:

$$
\boxed{
d_{\mathrm{Cu}}
\longrightarrow
d_{\mathrm{outer}}
\longrightarrow
\begin{cases}
N_{\mathrm{per\ layer}}\\
n_{\mathrm{layers}}
\end{cases}
\longrightarrow
N
}
$$

More explicitly:

$$
d_{\mathrm{Cu}}
\longrightarrow
d_{\mathrm{outer}}
$$

$$
\begin{aligned}
h,\ d_{\mathrm{outer}}
&\longrightarrow
N_{\mathrm{per\ layer}}
\\[6pt]
R-r,\ d_{\mathrm{outer}}
&\longrightarrow
n_{\mathrm{layers}}
\end{aligned}
$$

and finally:

$$
N_{\mathrm{per\ layer}},
\ n_{\mathrm{layers}}
\longrightarrow
N.
$$

#### From wire diameter to number of turns

The geometric relationship can be represented as:

$$
\boxed{
d_{\mathrm{Cu}}
\longrightarrow
d_{\mathrm{outer}}
\longrightarrow
\begin{cases}
N_{\mathrm{per\ layer}}\\
n_{\mathrm{layers}}
\end{cases}
\longrightarrow
N
}
$$

More explicitly:

$$
d_{\mathrm{Cu}}
\longrightarrow
d_{\mathrm{outer}}
$$

$$
\begin{aligned}
h,\ d_{\mathrm{outer}}
&\longrightarrow
N_{\mathrm{per\ layer}}
\\[6pt]
R-r,\ d_{\mathrm{outer}}
&\longrightarrow
n_{\mathrm{layers}}
\end{aligned}
$$

and finally:

$$
N_{\mathrm{per\ layer}},
\ n_{\mathrm{layers}}
\longrightarrow
N.
$$

In [11]:
import pandas as pd

wire_data = {
    "d_Cu_mm": [
        0.036,
        0.038,
        0.040,
        0.043,
        0.045,
        0.048,
        0.050,
        0.053,
        0.056,
        0.060,
        0.063,
        0.067,
        0.070,
        0.071,
        0.075,
        0.080,
        0.085,
        0.090,
        0.095,
        0.100,
        0.106,
        0.110,
        0.112,
        0.118,
        0.120,
        0.125,
        0.130,
        0.132,
        0.140,
        0.150,
        0.160,
        0.170,
        0.180,
        0.190,
        0.200,
        0.212,
        0.224,
        0.236,
        0.250,
        0.265,
        0.280,
        0.300,
        0.315,
        0.335,
        0.355,
        0.375,
        0.400,
        0.425,
        0.450,
        0.475,
        0.500,
    ],

    "area_Cu_mm2": [
        0.00101788,
        0.001134,
        0.001257,
        0.001452,
        0.001590,
        0.001810,
        0.001963,
        0.002206,
        0.002463,
        0.002827,
        0.003117,
        0.003526,
        0.003848,
        0.003959,
        0.004418,
        0.005027,
        0.005675,
        0.006362,
        0.007088,
        0.007854,
        0.008825,
        0.009503,
        0.009852,
        0.010936,
        0.011310,
        0.012272,
        0.013273,
        0.013685,
        0.015394,
        0.017671,
        0.020106,
        0.022698,
        0.025447,
        0.028353,
        0.031416,
        0.035299,
        0.039408,
        0.043744,
        0.049087,
        0.055155,
        0.061575,
        0.070686,
        0.077931,
        0.088141,
        0.098980,
        0.110447,
        0.125664,
        0.141863,
        0.159043,
        0.177205,
        0.196350,
    ],

    "d_outer_g1_min_mm": [
        0.040,
        0.042,
        0.044,
        0.047,
        0.050,
        0.053,
        0.055,
        0.058,
        0.062,
        0.066,
        0.069,
        0.074,
        0.077,
        0.078,
        0.082,
        0.087,
        0.093,
        0.098,
        0.103,
        0.108,
        0.115,
        0.119,
        0.121,
        0.128,
        0.130,
        0.135,
        0.141,
        0.143,
        0.151,
        0.162,
        0.172,
        0.183,
        0.193,
        0.204,
        0.214,
        0.227,
        0.239,
        0.253,
        0.267,
        0.283,
        0.298,
        0.319,
        0.334,
        0.355,
        0.375,
        0.396,
        0.421,
        0.447,
        0.472,
        0.499,
        0.524,
    ],

    "d_outer_g1_max_mm": [
        0.044,
        0.046,
        0.049,
        0.052,
        0.055,
        0.059,
        0.060,
        0.064,
        0.067,
        0.072,
        0.076,
        0.080,
        0.083,
        0.084,
        0.089,
        0.094,
        0.100,
        0.105,
        0.111,
        0.117,
        0.123,
        0.128,
        0.130,
        0.136,
        0.138,
        0.144,
        0.150,
        0.152,
        0.160,
        0.171,
        0.182,
        0.194,
        0.204,
        0.216,
        0.226,
        0.240,
        0.252,
        0.267,
        0.281,
        0.297,
        0.312,
        0.334,
        0.349,
        0.372,
        0.392,
        0.414,
        0.439,
        0.466,
        0.491,
        0.519,
        0.544,
    ],

    "R20_nom_ohm_m": [
        16.79,
        15.07,
        13.60,
        11.770,
        10.750,
        9.447,
        8.706,
        7.748,
        6.940,
        6.046,
        5.484,
        4.848,
        4.442,
        4.318,
        3.869,
        3.401,
        3.012,
        2.687,
        2.412,
        2.176,
        1.937,
        1.799,
        1.735,
        1.563,
        1.511,
        1.393,
        1.288,
        1.249,
        1.110,
        0.9673,
        0.8502,
        0.7531,
        0.6718,
        0.6029,
        0.5441,
        0.4843,
        0.4338,
        0.3908,
        0.3482,
        0.3099,
        0.2776,
        0.2418,
        0.2193,
        0.1939,
        0.1727,
        0.1548,
        0.1360,
        0.1205,
        0.1075,
        0.09646,
        0.08706,
    ],
}

wires = pd.DataFrame(wire_data)

wires.head()

,d_Cu_mm,area_Cu_mm2,d_outer_g1_min_mm,d_outer_g1_max_mm,R20_nom_ohm_m
0,0.036,0.001018,0.040,0.044,16.79
1,0.038,0.001134,0.042,0.046,15.07
2,0.040,0.001257,0.044,0.049,13.60
3,0.043,0.001452,0.047,0.052,11.77
4,0.045,0.001590,0.050,0.055,10.75


In [12]:
def calculate_number_of_turns(h, r, R, d_outer):
    """
    Calculate the approximate number of turns of a cylindrical coil.

    Parameters
    ----------
    h: float
       Axial length of the coil [mm].
    r: float
       Inner radius of the winding [mm].
    R: float
       Outer radius fo the winding [mm].
    d_outer: float
       Overall diameter of the enamelled wire [mm].

    Returns
    ----------
    N: int
       Approximate total number of turns.
    """
    turns_per_layer = int(h // d_outer)
    number_of_layers = int((R - r) // d_outer)

    N = turns_per_layer * number_of_layers

    return N

In [13]:
# A first test.
h = 10.0 # mm
r = 5.0 # mm
R = 7.0 # mm
d_outer = 0.544 # mm

N = calculate_number_of_turns(h, r, R, d_outer)

N

54

In [14]:
N_approx = h * (R - r) / d_outer ** 2

N_approx
# Here we have used division (/) instead of the integer division operator (//), used in the function.

67.58217993079585

In [15]:
# DataFrame test
wire = wires.iloc[34]

wire

d_Cu_mm              0.200000
area_Cu_mm2          0.031416
d_outer_g1_min_mm    0.214000
d_outer_g1_max_mm    0.226000
R20_nom_ohm_m        0.544100
Name: 34, dtype: float64

In [16]:
d_outer = (
    wire["d_outer_g1_min_mm"] + wire["d_outer_g1_max_mm"]
) / 2

d_outer

np.float64(0.22)

In [17]:
N = calculate_number_of_turns(
    h = 10.0,
    r = 5.0,
    R = 7.0,
    d_outer = d_outer
)

N

405

## Deepening: Wheeler's empirical formulas for inductance

Wheeler's empirical formulas can be used to estimate the inductance
of coils from their geometry and number of turns.

In general, the inductance can be expressed as:

$$
\mathcal{L}
=
f(N,\text{geometry},\mu)
$$

where:

- $\mathcal{L}$ is the inductance;
- $N$ is the number of turns;
- $\text{geometry}$ describes the dimensions of the coil;
- $\mu$ represents the magnetic permeability of the magnetic circuit.

For many coil geometries, Wheeler's formulas have a dependence on
the square of the number of turns:

$$
\mathcal{L}\propto N^2.
$$

These formulas provide an empirical estimate of the inductance of
a real coil and could therefore be integrated into the model as a
physics-based derived quantity.

At this stage, however, inductance is considered a possible future
extension of the model, since no real inductance measurements are
currently available.

For a single-layer air core solenoid, the inductance can be estimated as:

$$
\boxed{
\mathcal{L}[\mu H]
=
\frac{d^2N^2}{18d+40l}
}
$$

where:

- $d$ is the coil diameter, expressed in mm;
- $l$ is the axial length of the coil, expressed in mm;
- $N$ is the number of turns.

For a multilayer air core solenoid, the inductance can be estimated as:

$$
\boxed{
\mathcal{L}[\mu H]
=
\frac{0.8d^2N^2}{6d+9l+10t}
}
$$

where:

- $d$ is the mean coil diameter, expressed in mm;
- $l$ is the axial length of the coil, expressed in mm;
- $t$ is the radial winding thickness, expressed in mm;
- $N$ is the total number of turns.

### Resistive and inductive properties

The same coil geometry and number of turns can determine both
resistive and inductive properties:

$$
\text{Coil geometry}
+
N
\longrightarrow
\begin{cases}
\text{Wire length }L \\
\text{Inductance }\mathcal{L}
\end{cases}
$$

The resistive branch is:

$$
L,
\ A_{\text{Cu}},
\ \rho(T)
\longrightarrow
R
$$

with:

$$
\boxed{
R=\rho(T)\frac{L}{A_{\text{Cu}}}
}
$$

The inductive branch is:

$$
N,
\ \text{coil geometry},
\ \mu
\longrightarrow
\mathcal{L}
$$

Therefore, conceptually:

$$
\boxed{
\text{Coil geometry},\,N
\longrightarrow
\begin{cases}
\text{Resistive properties}\\
\text{Inductive properties}
\end{cases}
}
$$